In [1]:
import os
import sys

import torch

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "info")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

_nb = os.getcwd()
if os.path.basename(_nb) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(_nb, ".."))
else:
    PROJECT_ROOT = os.environ.get(
        "AUDIO_STREAM_ADAPTER_ROOT",
        os.path.abspath(os.path.join(_nb, "..")),
    )
SRC_ROOT = os.path.join(PROJECT_ROOT, "src")
for _p in (SRC_ROOT, PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from transformers import AutoModelForCausalLM, AutoTokenizer

# Streaming Audio Encoder

In [2]:
# src/utils/qwen_model_loader.py

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

if device == "cuda":
    torch.cuda.init()

print(f"Using device: {device}")

MODEL_ID = "Qwen/Qwen3-8B"

# Load tokenizer and model
qwen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

_load_kw = dict(torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True)
if device == "cuda":
    _load_kw["device_map"] = "auto"

qwen_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_load_kw)
if device == "cuda" and "device_map" not in _load_kw:
    qwen_model = qwen_model.to(device)

qwen_model.eval()

print("Qwen3-8B model ready!")


Using device: cuda


loading configuration file config.json from cache at /home/ml/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218/config.json
Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attenti

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

loading configuration file generation_config.json from cache at /home/ml/.cache/huggingface/hub/models--Qwen--Qwen3-8B/snapshots/b968826d9c46dd6066d109eabc6255188de91218/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}

Some parameters are on the meta device because they were offloaded to the cpu.


Qwen3-8B model ready!


In [3]:
# src/utils/whisper_model_loader.py

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import pipeline

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {device}")

MODEL_ID = "openai/whisper-small"

# Load processor and model
whisper_processor = WhisperProcessor.from_pretrained(MODEL_ID)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
whisper_model.to(device)

# Build inference pipeline
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

print(type(whisper_model))  # should output: <class 'transformers.models.whisper.modeling_whisper.WhisperForConditionalGeneration'>
print("Whisper small model ready!")

Using device: cuda


loading configuration file processor_config.json from cache at None
loading configuration file preprocessor_config.json from cache at /home/ml/.cache/huggingface/hub/models--openai--whisper-small/snapshots/973afd24965f72e36ca33b3055d56a652f456b4d/preprocessor_config.json
loading configuration file preprocessor_config.json from cache at /home/ml/.cache/huggingface/hub/models--openai--whisper-small/snapshots/973afd24965f72e36ca33b3055d56a652f456b4d/preprocessor_config.json
Feature extractor WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

loading configuration file config.json from cache at /home/ml/.cache/huggingface/hub/models--openai--whisper-small/snapshots/973afd24965f72e36ca33b3055d56a652f456b4d/

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

loading configuration file generation_config.json from cache at /home/ml/.cache/huggingface/hub/models--openai--whisper-small/snapshots/973afd24965f72e36ca33b3055d56a652f456b4d/generation_config.json
Generate config GenerationConfig {
  "alignment_heads": [
    [
      5,
      3
    ],
    [
      5,
      9
    ],
    [
      8,
      0
    ],
    [
      8,
      4
    ],
    [
      8,
      7
    ],
    [
      8,
      8
    ],
    [
      9,
      0
    ],
    [
      9,
      7
    ],
    [
      9,
      9
    ],
    [
      10,
      5
    ]
  ],
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "decoder_start_token_id": 50258,
  "eos_token_id": 50257,
  "forced_decoder_ids": [
    [
      1,
      null
    ],
    [
      2,
      50359
    ]
  ],
  "is_multilingual": true,
  "lang_to_id": {
    "<|af|>": 50327,
    "<|am|>": 50334,
    "<|ar|>": 50272,
    "<|as|>": 50350,
    "<|az|>": 50304,
    "<|ba|>": 50355,
    "<|be|>": 50330,
    "<|bg|

<class 'transformers.models.whisper.modeling_whisper.WhisperForConditionalGeneration'>
Whisper small model ready!


## Transcription Summarization

In [4]:
print(os.getcwd())

/home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter/notebooks


In [5]:
# !ls -l ../datasets/librispeech_data/LibriSpeech/train-clean-100

In [6]:
import glob

dataset_root = "../datasets/librispeech_data/LibriSpeech/train-clean-100"
flac_files = sorted(glob.glob(f"{dataset_root}/**/*.flac", recursive=True))
print(f"Found {len(flac_files)} files\n")

Found 28539 files



### Transcribe (transcribe.py)

In [7]:
# transcribe_wav and transcribe_dataset_stream
import librosa
from IPython.display import Audio, display

# import pipe from src/utils/whisper_model_loader.py
# from src.utils.whisper_model_loader import pipe

SAMPLES_RATE = 16000

def transcribe_wav(file_path:str) -> str:
    waveform, sample_rate = librosa.load(file_path, sr=None)
    if len(waveform.shape) > 1:
        waveform = librosa.to_mono(waveform)
    if sample_rate != SAMPLES_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLES_RATE)
    result = whisper_pipe(
        waveform,
        generate_kwargs={"language": "en", "task": "transcribe"},
    )
    return result["text"], waveform, sample_rate

def transcribe_dataset_stream(dataset_root: str, num_samples: int = 5):
    """
    transcribes files one at a time and yields each result immediately as {"file": path, "transcription": text}.

    start processing results as soon as they arrive, without waiting for all files to finish.
    """
    flac_files = sorted(glob.glob(f"{dataset_root}/**/*.flac", recursive=True))
    print(f"Found {len(flac_files)} files\n")
    count = 0

    for path in flac_files:
        if count < num_samples:
            transcription, waveform, original_sample_rate = transcribe_wav(path)
            # print(f"{path} → {transcription}\n")
            count = count + 1
            yield {"file": path, "transcription": transcription, "waveform": waveform}

In [8]:
def transcribe_dataset(dataset_root: str) -> list[dict]:
    """
    Eagerly transcribes all files and returns a list of
    {"file": path, "transcription": text} dicts.

    Use transcribe_dataset_stream() instead if you want to process
    results incrementally as each file is transcribed.
    """
    return list[dict[str, str]](transcribe_dataset_stream(dataset_root))

# results = transcribe_dataset(dataset_root)
# results[0], display(Audio(results[0]["waveform"], rate=SAMPLES_RATE))

## Q-former Adapter

In [9]:
#  src/adapter/cross_attention.py
import math

class QFormerLayer(nn.Module):
    """
    Single Q-Former layer following BLIP-2 architecture.

    Sub-layer 1 — Self-Attention:
        Queries attend to each other. This is what makes Q-Former different
        from plain cross-attention. Without it, each query works independently
        and they may extract redundant information. With self-attention,
        Query 0 can see what Query 1 is capturing and focus elsewhere.

    Sub-layer 2 — Cross-Attention:
        Queries attend to encoder frames (Whisper output). This is where
        the actual information extraction happens.

    Sub-layer 3 — Feed-Forward Network:
        Standard transformer FFN for per-token nonlinear transformation.

    Args:
        d_model: Dimension of queries and output.
        num_heads: Number of attention heads (shared across self and cross attn).
        d_ffn: Hidden dimension of the feed-forward network.
        dropout: Dropout rate.
        use_cross_attention: If False, only self-attention + FFN (no audio cross-attn).
    """

    def __init__(
        self,
        d_model: int = 1024,
        num_heads: int = 4,
        d_ffn: int = 2048,
        dropout: float = 0.1,
        use_cross_attention: bool = True,
    ):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.scale = math.sqrt(self.d_head)
        self.use_cross_attention = use_cross_attention

        # ---- Sub-layer 1: Self-Attention (queries ↔ queries) ----
        self.self_attn_q = nn.Linear(d_model, d_model)
        self.self_attn_k = nn.Linear(d_model, d_model)
        self.self_attn_v = nn.Linear(d_model, d_model)
        self.self_attn_o = nn.Linear(d_model, d_model)
        self.norm_self = nn.LayerNorm(d_model)
        self.self_attn_dropout = nn.Dropout(dropout)

        # ---- Sub-layer 2: Cross-Attention (queries → encoder frames) ----
        if use_cross_attention:
            self.cross_attn_q = nn.Linear(d_model, d_model)
            self.cross_attn_k = nn.Linear(d_model, d_model)
            self.cross_attn_v = nn.Linear(d_model, d_model)
            self.cross_attn_o = nn.Linear(d_model, d_model)
            self.norm_cross_q = nn.LayerNorm(d_model)
            self.norm_cross_kv = nn.LayerNorm(d_model)
            self.cross_attn_dropout = nn.Dropout(dropout)

        # ---- Sub-layer 3: Feed-Forward Network ----
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model),
            nn.Dropout(dropout),
        )
        self.norm_ffn = nn.LayerNorm(d_model)

    def _multihead_attention(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        dropout: nn.Dropout,
    ) -> torch.Tensor:
        """
        Shared multi-head attention logic for both self and cross attention.

        Args:
            q: (batch, seq_q, d_model) — already projected queries
            k: (batch, seq_k, d_model) — already projected keys
            v: (batch, seq_k, d_model) — already projected values
            dropout: dropout module for attention weights

        Returns:
            (batch, seq_q, d_model)
        """
        batch_size, seq_q, _ = q.shape
        seq_k = k.shape[1]

        # Reshape to multi-head: (batch, seq, d_model) → (batch, heads, seq, d_head)
        q = q.view(batch_size, seq_q, self.num_heads, self.d_head).transpose(1, 2)
        k = k.view(batch_size, seq_k, self.num_heads, self.d_head).transpose(1, 2)
        v = v.view(batch_size, seq_k, self.num_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = dropout(attn_weights)

        # Weighted sum and merge heads
        output = torch.matmul(attn_weights, v)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_q, self.d_model)

        return output

    def forward(
        self,
        queries: torch.Tensor,
        encoder_features: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            queries: (batch, m, d_model) — learnable query vectors
            encoder_features: (batch, T, d_model) — Whisper encoder output

        Returns:
            (batch, m, d_model) — updated query representations
        """
        # ---- Sub-layer 1: Self-Attention (queries attend to each other) ----
        q_norm = self.norm_self(queries)
        sa_out = self._multihead_attention(
            q=self.self_attn_q(q_norm),
            k=self.self_attn_k(q_norm),
            v=self.self_attn_v(q_norm),
            dropout=self.self_attn_dropout,
        )
        sa_out = self.self_attn_o(sa_out)
        queries = queries + sa_out  # residual

        # ---- Sub-layer 2: Cross-Attention (queries attend to encoder) ----
        if self.use_cross_attention:
            q_norm = self.norm_cross_q(queries)
            kv_norm = self.norm_cross_kv(encoder_features)
            ca_out = self._multihead_attention(
                q=self.cross_attn_q(q_norm),
                k=self.cross_attn_k(kv_norm),
                v=self.cross_attn_v(kv_norm),
                dropout=self.cross_attn_dropout,
            )
            ca_out = self.cross_attn_o(ca_out)
            queries = queries + ca_out  # residual

        # ---- Sub-layer 3: FFN ----
        queries = queries + self.ffn(self.norm_ffn(queries))  # residual

        return queries

## Stability Buffer

In [10]:
class StabilityBuffer(nn.Module):
    """
    EMA-based temporal smoothing for streaming token sequences.

    For each new window's tokens Z_t, the smoothed output is:
        Z'_t = alpha * Z_t + (1 - alpha) * Z'_{t-1}

    Args:
        alpha: EMA smoothing factor in (0, 1]. Higher = more weight on current window.
               0.8 means current window contributes 80%, history contributes 20%.
        learnable: If True, alpha is a trainable parameter (initialized to init value).
    """

    def __init__(self, alpha: float = 0.8, learnable: bool = False):
        super().__init__()
        if learnable:
            # Store in logit space so sigmoid keeps it in (0, 1)
            alpha_logit = torch.log(torch.tensor(alpha / (1.0 - alpha)))
            self._alpha_logit = nn.Parameter(alpha_logit)
        else:
            self.register_buffer("_alpha", torch.tensor(alpha))

        self.learnable = learnable
        # Buffer state for streaming inference (not a parameter, not saved in state_dict)
        self._prev_tokens: torch.Tensor | None = None

    @property
    def alpha(self) -> torch.Tensor:
        if self.learnable:
            return torch.sigmoid(self._alpha_logit)
        return self._alpha

    def reset(self):
        """Reset buffer state. Call at the start of each new audio stream."""
        self._prev_tokens = None

    def forward(
        self,
        tokens: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Apply EMA smoothing and compute stability loss.

        Args:
            tokens: (batch, m, d) — adapter output for current window

        Returns:
            smoothed: (batch, m, d) — EMA-smoothed tokens
            stability_loss: scalar — ||Z_t - Z_{t-1}||^2, zero for first window
        """
        if self._prev_tokens is None:
            # First window: no history to smooth against
            self._prev_tokens = tokens.detach()
            zero_loss = torch.tensor(0.0, device=tokens.device, dtype=tokens.dtype)
            return tokens, zero_loss

        # Stability loss: penalize large jumps between adjacent windows
        # Compute in float32 for numerical stability with float16 models
        stability_loss = torch.mean((tokens - self._prev_tokens).float() ** 2)

        # EMA smoothing
        alpha = self.alpha
        smoothed = alpha * tokens + (1.0 - alpha) * self._prev_tokens

        # Update buffer (detach to prevent backprop through time)
        self._prev_tokens = smoothed.detach()

        return smoothed, stability_loss

    def forward_sequence(
        self,
        token_sequence: list[torch.Tensor],
    ) -> tuple[list[torch.Tensor], torch.Tensor]:
        """
        Process a full sequence of windows (for training on complete utterances).

        Args:
            token_sequence: List of (batch, m, d) tensors, one per window

        Returns:
            smoothed_sequence: List of smoothed tensors
            total_stability_loss: Sum of per-step stability losses
        """
        self.reset()
        smoothed = []
        total_loss = torch.tensor(0.0, device=token_sequence[0].device)

        for tokens in token_sequence:
            s, loss = self.forward(tokens)
            smoothed.append(s)
            total_loss = total_loss + loss

        return smoothed, total_loss


## Adaptive Rate Controller

In [11]:
class AdaptiveRateController(nn.Module):
    """
    Produces per-query gate scores based on input complexity.

    Architecture:
        mean_pool(F) -> Linear -> ReLU -> Linear -> sigmoid -> gate per query

    During training: soft gates (multiply token embeddings by gate scores)
    During inference: hard gates (drop tokens with gate < threshold)

    Args:
        d_encoder: Dimension of encoder features (Whisper output dim).
        m_max: Maximum number of query tokens.
        hidden_dim: Hidden dimension of the gating MLP.
        threshold: Hard gate threshold for inference.
        target_rate: Target average number of active tokens per window.
                     Used for L_rate computation (e.g., 2.0 means aim for 2 tokens/window).
    """

    def __init__(
        self,
        d_encoder: int = 1024,
        m_max: int = 4,
        hidden_dim: int = 256,
        threshold: float = 0.5,
        target_rate: float = 2.0,
    ):
        super().__init__()
        self.m_max = m_max
        self.threshold = threshold
        self.target_rate = target_rate

        self.gate_mlp = nn.Sequential(
            nn.Linear(d_encoder, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, m_max),
        )

    def forward(
        self,
        encoder_features: torch.Tensor,
        tokens: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        """
        Compute gated tokens based on input complexity.

        Args:
            encoder_features: (batch, T, d_encoder) -- Whisper frame features
            tokens: (batch, m, d) -- adapter output tokens

        Returns:
            dict with:
                tokens: (batch, m, d) -- gated tokens
                gate_scores: (batch, m) -- gate values per query slot
                sparse_loss: scalar -- L_sparse (encourages fewer active tokens)
                rate_loss: scalar -- L_rate (penalizes deviation from target rate)
        """
        # Pool encoder features to get a single complexity vector
        pooled = encoder_features.mean(dim=1)  # (batch, d_encoder)

        # Compute per-query gate scores
        gate_scores = torch.sigmoid(self.gate_mlp(pooled))  # (batch, m_max)

        if self.training:
            # Soft gating: multiply tokens by gate scores (keeps gradients flowing)
            gated = tokens * gate_scores.unsqueeze(-1)  # (batch, m, d)
        else:
            # Hard gating: zero out tokens below threshold
            mask = (gate_scores > self.threshold).unsqueeze(-1)  # (batch, m, 1)
            gated = tokens * mask.float()

        # L_sparse: encourage sparsity (L1 on gate scores)
        # Lower gate scores = fewer active tokens = more compression
        sparse_loss = gate_scores.mean()

        # L_rate: penalize deviation from target token rate
        # Effective token count = sum of gate scores (soft count)
        effective_count = gate_scores.sum(dim=-1)  # (batch,)
        rate_loss = torch.mean((effective_count - self.target_rate) ** 2)

        return {
            "tokens": gated,
            "gate_scores": gate_scores,
            "sparse_loss": sparse_loss,
            "rate_loss": rate_loss,
        }


## Streaming Adapter

In [12]:
class StreamingAdapter(nn.Module):
    """
    Component 2: Trainable streaming adapter network.

    Q-Former style cross-attention resampler with stability buffer
    and optional adaptive token rate controller.

    Args:
        d_encoder: Whisper encoder output dimension (1024 for whisper-medium).
        d_llm: Target LLM embedding dimension.
        num_queries: Maximum number of compressed tokens per window (m=1-4).
        num_layers: Number of stacked Q-Former layers.
        num_heads: Number of attention heads per Q-Former layer.
        d_ffn: FFN hidden dimension in Q-Former layers.
        dropout: Dropout rate.
        ema_alpha: EMA smoothing factor for stability buffer.
        learnable_ema: Whether EMA alpha is trainable.
        use_rate_controller: Whether to use adaptive token rate control.
                             Set False for initial experiments (fixed m tokens/window).
        rate_threshold: Hard gate threshold for inference (if rate controller enabled).
        target_rate: Target average tokens per window (if rate controller enabled).
        cross_layer_in_between: Number of self-attention-only layers (self + FFN, no
            cross-attention to audio) between successive cross-attention layers.
            0 means every layer includes cross-attention (full Q-Former stack).
            For K > 0, let P = K + 1. Cross-attention runs at the *end* of each block of
            P layers: layer index i uses cross-attention iff i % P == P - 1 (e.g. K=1 →
            cross on layers 1, 3, 5, … and self-only on 0, 2, 4, … so the stack does not
            start with cross-attention).
    """

    def __init__(
        self,
        d_encoder: int = 1024,
        d_llm: int = 2560,
        num_queries: int = 4,
        num_layers: int = 2,
        num_heads: int = 4,
        d_ffn: int = 2048,
        dropout: float = 0.1,
        ema_alpha: float = 0.8,
        learnable_ema: bool = False,
        use_rate_controller: bool = False,
        rate_threshold: float = 0.5,
        target_rate: float = 2.0,
        cross_layer_in_between: int = 1,
    ):
        super().__init__()
        if cross_layer_in_between < 0:
            raise ValueError("cross_layer_in_between must be >= 0")
        self.d_encoder = d_encoder
        self.d_llm = d_llm
        self.num_queries = num_queries
        self.use_rate_controller = use_rate_controller
        self.cross_layer_in_between = cross_layer_in_between

        # Learnable query vectors Q ∈ R^{m × D_q}
        self.queries = nn.Parameter(torch.randn(1, num_queries, d_encoder) * 0.02)

        # # Stack of Q-Former layers (self-attn + cross-attn + FFN, BLIP-2 style)
        # self.layers = nn.ModuleList([
        #     QFormerLayer(
        #         d_model=d_encoder,
        #         num_heads=num_heads,
        #         d_ffn=d_ffn,
        #         dropout=dropout,
        #     )
        #     for _ in range(num_layers)
        # ])

        # Stacked layers: optional self-only layers between cross-attention layers.
        # period P = K+1: cross at i ≡ P-1 (mod P) — last slot in each block, so layer 0
        # is self-only when K>0 (e.g. K=1 → cross on 1,3,5,... not 0,2,4,...).
        period = cross_layer_in_between + 1
        self.layers = nn.ModuleList([
            QFormerLayer(
                d_model=d_encoder,
                num_heads=num_heads,
                d_ffn=d_ffn,
                dropout=dropout,
                use_cross_attention=(i % period == period - 1),
            )
            for i in range(num_layers)
        ])


        # Project from encoder space to LLM embedding space
        self.output_proj = nn.Sequential(
            nn.LayerNorm(d_encoder),
            nn.Linear(d_encoder, d_llm),
        )

        # Optional: Adaptive rate controller
        self.rate_controller = None
        if use_rate_controller:
            self.rate_controller = AdaptiveRateController(
                d_encoder=d_encoder,
                m_max=num_queries,
                threshold=rate_threshold,
                target_rate=target_rate,
            )

        # Stability buffer for temporal consistency
        self.stability_buffer = StabilityBuffer(
            alpha=ema_alpha,
            learnable=learnable_ema,
        )

    def reset_streaming_state(self):
        """Reset buffer state. Call at the start of each new audio stream."""
        self.stability_buffer.reset()

    def forward_window(
        self,
        encoder_features: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        """
        Process a single audio window (streaming inference mode).

        Args:
            encoder_features: (batch, T, d_encoder) -- Whisper encoder output
                              for one overlapping window (0.8s of audio)

        Returns:
            dict with:
                tokens: (batch, m, d_llm) -- compressed, smoothed tokens
                stability_loss: scalar -- L_stability (temporal consistency)
                gate_scores: (batch, m) -- rate gate values (None if no rate controller)
                sparse_loss: scalar -- L_sparse (None if no rate controller)
                rate_loss: scalar -- L_rate (None if no rate controller)
        """
        batch_size = encoder_features.shape[0]

        # Expand learnable queries to batch size
        q = self.queries.expand(batch_size, -1, -1)  # (batch, m, d_encoder)

        # Pass through Q-Former layers (self-attn → cross-attn → FFN per layer)
        for layer in self.layers:
            q = layer(q, encoder_features)

        # Optional: adaptive rate control
        gate_scores = None
        sparse_loss = None
        rate_loss = None
        if self.rate_controller is not None:
            rc_result = self.rate_controller(encoder_features, q)
            q = rc_result["tokens"]
            gate_scores = rc_result["gate_scores"]
            sparse_loss = rc_result["sparse_loss"]
            rate_loss = rc_result["rate_loss"]

        # Project to LLM dimension
        z = self.output_proj(q)  # (batch, m, d_llm)

        # Temporal smoothing via stability buffer
        z_smooth, stability_loss = self.stability_buffer(z)

        return {
            "tokens": z_smooth,
            "stability_loss": stability_loss,
            "gate_scores": gate_scores,
            "sparse_loss": sparse_loss,
            "rate_loss": rate_loss,
        }

    def forward(
        self,
        encoder_features_sequence: list[torch.Tensor],
    ) -> dict[str, torch.Tensor]:
        """
        Process a full sequence of windows (training mode).

        Args:
            encoder_features_sequence: List of (batch, T, d_encoder) tensors,
                one per overlapping window from a complete utterance.

        Returns:
            dict with:
                tokens: (batch, total_tokens, d_llm) -- all tokens concatenated
                stability_loss: scalar -- total L_stability across all windows
                gate_scores: (batch, num_windows, m) or None
                sparse_loss: scalar or None -- total L_sparse
                rate_loss: scalar or None -- total L_rate
        """
        self.reset_streaming_state()
        device = encoder_features_sequence[0].device
        dtype = encoder_features_sequence[0].dtype

        all_tokens = []
        all_gates = []
        total_stability = torch.tensor(0.0, device=device, dtype=dtype)
        total_sparse = torch.tensor(0.0, device=device, dtype=dtype)
        total_rate = torch.tensor(0.0, device=device, dtype=dtype)

        for window_features in encoder_features_sequence:
            result = self.forward_window(window_features)
            all_tokens.append(result["tokens"])
            total_stability = total_stability + result["stability_loss"]

            if result["gate_scores"] is not None:
                all_gates.append(result["gate_scores"])
                total_sparse = total_sparse + result["sparse_loss"]
                total_rate = total_rate + result["rate_loss"]

        tokens = torch.cat(all_tokens, dim=1)  # (batch, num_windows * m, d_llm)

        gate_scores = None
        sparse_loss = None
        rate_loss = None
        if all_gates:
            gate_scores = torch.stack(all_gates, dim=1)  # (batch, num_windows, m)
            sparse_loss = total_sparse
            rate_loss = total_rate

        return {
            "tokens": tokens,
            "stability_loss": total_stability,
            "gate_scores": gate_scores,
            "sparse_loss": sparse_loss,
            "rate_loss": rate_loss,
        }


In [13]:
# Frames → windows: WhisperFrameWindowizer lives in src/adapter/windowing.py
import time
import math
import matplotlib.pyplot as plt


In [14]:
# Add src/ so `import adapter` resolves (package lives under audio-streaming-adapter/src)
import sys
import os


_SRC = os.path.join(PROJECT_ROOT, "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from adapter import StreamingAdapter, WhisperFrameWindowizer
from adapter_llm_pipeline import WhisperAdapterLLMPipeline

# Default: 0.8s window, 0.4s stride (50% overlap) → 1500 frames → many overlapping windows
windowizer = WhisperFrameWindowizer(window_seconds=0.8, stride_seconds=0.4)

print("dim of qwen_model.config.hidden_size: ", qwen_model.config.hidden_size)

streaming_adapter = StreamingAdapter(
    d_encoder=768,
    d_llm=qwen_model.config.hidden_size,
    num_queries=32,
    num_layers=8,
    num_heads=12,
    d_ffn=2048,
    dropout=0.1,
    use_rate_controller=True,
).to(device=device, dtype=torch_dtype)


# Optional: load a trained adapter checkpoint
# Accepts checkpoints produced by stage scripts or the stage notebooks.
import os
import torch

CHECKPOINT_PATH = os.environ.get("ADAPTER_CHECKPOINT_PATH", "")  # e.g. 'checkpoints/adapter_adapter.pt'
if CHECKPOINT_PATH:
    if not os.path.isabs(CHECKPOINT_PATH):
        # training checkpoints default to <package>/checkpoints/ when running stage scripts from package root
        candidate = os.path.join(PROJECT_ROOT, CHECKPOINT_PATH)
        CHECKPOINT_PATH = candidate

    ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
    state = (
        ckpt.get("adapter_state_dict")
        or ckpt.get("model_state_dict")
        or ckpt.get("state_dict")
    )
    if state is None:
        raise KeyError(f"No adapter state_dict found in checkpoint keys: {list(ckpt.keys())}")

    missing, unexpected = streaming_adapter.load_state_dict(state, strict=False)
    print("Loaded adapter checkpoint:", CHECKPOINT_PATH)
    print("missing:", len(missing), "unexpected:", len(unexpected))

pipeline = WhisperAdapterLLMPipeline(
    whisper_processor=whisper_processor,
    whisper_model=whisper_model,
    windowizer=windowizer,
    streaming_adapter=streaming_adapter,
    llm_model=qwen_model,
    llm_tokenizer=qwen_tokenizer,
    device=device,
    torch_dtype=torch_dtype,
)

print("pipeline:", pipeline)

for entry in transcribe_dataset_stream(dataset_root, num_samples=100):
    waveform = entry["waveform"]

    # n_windows: how many overlapping windows (from this encode) feed the adapter for ONE LLM call.
    # n_windows=-1 → use every window (long inputs_embeds; ~74 windows × 32 queries for 1500 frames).
    out = pipeline.generate(
        waveform,
        n_windows=-1,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
    )

    print("num_windows_used:", out["num_windows_used"])
    print("adapter tokens:", out["adapter_out"]["tokens"].shape)
    print("encode_time_s:", out["encode_time_s"])
    print("enc.shape:", out["enc"].shape, "windows.shape:", out["windows"].shape)
    print("Summary:", out["text"])
    break

# Examples:
# out = pipeline.generate(waveform, n_windows=3, max_new_tokens=100)   # first 3 windows only
# out = pipeline.generate(waveform, n_windows=-1, max_new_tokens=128)  # all windows from encoder


dim of qwen_model.config.hidden_size:  4096


pipeline: <adapter_llm_pipeline.WhisperAdapterLLMPipeline object at 0x7e260bb2eed0>
Found 28539 files



Increase max_length from 448 to 448 since input is conditioned on previous segment.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginL

num_windows_used: 74
adapter tokens: torch.Size([1, 2368, 4096])
encode_time_s: 0.009131669998168945
enc.shape: torch.Size([1, 1500, 768]) windows.shape: torch.Size([1, 74, 40, 768])
Summary: It seems like the input you provided is not a valid audio file or transcript. The text appears to be a series of repeated Chinese characters, which might be a result of an encoding error or a corrupted file. I cannot provide a summary of the audio content in this case. Please ensure that you provide a valid audio file or a proper transcript for accurate assistance.
